### Agent接入MCP服务

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi

load_dotenv()

# 创建大语言模型实例
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.5,
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    # streaming=True,
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_30968\467523796.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_models import ChatTongyi


In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient(
    {
        "amap-maps-streamableHTTP": {
            "url": "https://mcp.amap.com/mcp?key=xxxxxxxxxxxxxxx",
            "transport": "streamable_http",
        },

        # 使用标准格式stdio传输，调用MCP服务。实际上就等于执行命令 npx @amap/amap-maps-mcp-server + AMAP_MAPS_API_KEY + 你的问题
        # "amap-maps": {
        #     "command": "npx",
        #     "args": ["-y", "@amap/amap-maps-mcp-server"],
        #     "env": {
        #         "AMAP_MAPS_API_KEY": "xxxxxxxxxxxxx"
        #     },
        #     "transport": "stdio",
        # }
    }
)

tools = await client.get_tools()
agent = create_agent(
    model=llm,
    tools=tools,
)
response = await agent.ainvoke(
    {"messages":[{"role": "user", "content": "帮我规划一条从西安钟楼到大雁塔的自驾路线"}]}
)
response

{'messages': [HumanMessage(content='帮我规划一条从西安钟楼到大雁塔的自驾路线', additional_kwargs={}, response_metadata={}, id='3aca39bc-9b8b-4553-91f9-d43db2aed96d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_567b47588d6e499789bd13', 'type': 'function', 'function': {'name': 'maps_geo', 'arguments': '{"address": "西安钟楼"}'}}, {'index': 1, 'id': 'call_0a360ca24a4d4624acb328', 'type': 'function', 'function': {'name': 'maps_geo', 'arguments': '{"address": "大雁塔"}'}}]}, response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'tool_calls', 'request_id': 'd5552923-9f74-9c34-bcd8-287f7036eb6d', 'token_usage': {'input_tokens': 2193, 'output_tokens': 43, 'total_tokens': 2236, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--01a135dd-606f-77c3-a498-1f015064d1bb-0', tool_calls=[{'name': 'maps_geo', 'args': {'address': '西安钟楼'}, 'id': 'call_567b47588d6e499789bd13', 'type': 'tool_call'}, {'name': 'maps_geo', 'args': {'address': '大雁塔'}, 'id': 'call_0a360ca24a4d4624

### 本地开发MCP服务

In [ ]:
from mcp.server.fastmcp import FastMCP
# from mcp.server import MCPServer


mcp = FastMCP("roymcpdemo")

@mcp.tool()
def add (a: int, b: int) -> int:
    """Add two numbers together."""
    print(f"roy mcp demo called: add({a}, {b})")
    return a + b

@mcp.tool()
def weather(city: str):
    """获取某个城市的天气信息

    Args:
        city (str): 城市名称
    Returns:
        str: 天气信息
       """
    return f"{city}的天气是晴朗的"

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """
    Greet a person by name.
    :param name: 用户姓名
    :return: 欢迎信息
       """
    print(f"roy mcp demo called: greeting({name})")
    return f"你好，{name}！"

if __name__ == "__main__":
    # 以http协议暴露服务
    mcp.run(transport="streamable-http")
    # 以stdio协议暴露服务
    # mcp.run(transport="stdio")